# Train FurFriends AI v7 on a free Colab GPU

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. Run the cells in order (click a cell, then Shift+Enter). It takes about 30–40 minutes in all.
3. The upload cell asks for `ai-server/colab/furfriends-train-v7.zip` from the team laptop (made by `python colab/make_bundle.py`).
4. The last cell downloads `rag-v7.zip`. Unzip it into `ai-server/adapters/` on the team laptop, so you get
   `ai-server/adapters/rag-v7/`, then follow the README's export and test steps.

Nothing here is uploaded anywhere except this Colab session. The base model (Qwen2.5-0.5B-Instruct, Apache 2.0) is
downloaded from Hugging Face.

In [ ]:
# 1. Check the GPU: it should say "Tesla T4". If it fails, do step 1 above.
!nvidia-smi -L

In [ ]:
# 2. The same versions as the laptop's export Python (C:\Python312), so the trained adapter exports without surprises.
!pip install -q transformers==4.57.0 peft==0.17.1 trl==0.23.1 accelerate==1.10.1 datasets==4.1.1

In [ ]:
# 3. Upload ai-server/colab/furfriends-train-v7.zip
import zipfile
from google.colab import files
up = files.upload()
zipfile.ZipFile(next(iter(up))).extractall(".")
!ls -la . data
!wc -l data/rag_train_v7.jsonl

In [ ]:
# 4. Train: all layers, 2 passes over the data, full precision (about 20-30 minutes on a T4).
!python train_lora.py --data data/rag_train_v7.jsonl --base Qwen/Qwen2.5-0.5B-Instruct --out adapters/rag-v7 --layers all --epochs 2 --max-length 1024

In [ ]:
# 5. Optional quick look: what the trained model writes for 3 training questions. They were trained on, so this only
#    shows the training worked; the real test (questions it never saw) runs on the laptop.
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from prompting import build_messages
base = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(base)
model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(base, dtype=torch.float32).cuda(), "adapters/rag-v7").eval()
rows = [json.loads(l) for l in open("data/rag_train_v7.jsonl", encoding="utf-8")]
for r in [r for r in rows if r.get("written_answer") and not r.get("pet")][:3]:
    ids = tok.apply_chat_template(build_messages(r["messages"][:-1], r.get("pet"), r.get("sources")),
                                  add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(ids, max_new_tokens=220, do_sample=False)
    print("Q:", r["messages"][-2]["content"])
    print("A:", tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
    print("-" * 60)

In [ ]:
# 6. Download the adapter (a few MB). Unzip it into ai-server/adapters/ on the team laptop.
!cd adapters && rm -rf rag-v7/checkpoint-* && zip -qr rag-v7.zip rag-v7 && ls -la rag-v7.zip
from google.colab import files
files.download("adapters/rag-v7.zip")